## My Edits

Imports

In [30]:
import torch
import torchvision
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.datasets import MNIST
from torchvision.datasets import FashionMNIST ## changing to mnist dataset
import os
import random

Output directory

In [31]:
if not os.path.exists('./results'):
    os.mkdir('./results')

Img Conversion helper

In [32]:
def to_img(x):
    x = 0.5 * (x + 1)
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 1, 28, 28)
    return x

Training hyperparameters

In [33]:
num_epochs = 100
batch_size = 128
learning_rate = 1e-3

Image preprocessing pipeline

In [34]:
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])


Dataset + DataLoader

split into patches of 128

In [35]:
dataset = FashionMNIST('./FashiondataWeek1', transform=img_transform, download=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

100.0%
100.0%
100.0%
100.0%


Autoencoder model

In [36]:
class autoencoder(nn.Module):
    def __init__(self):
        super(autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=3, padding=1),  # b, 16, 10, 10 layer 1
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),  # b, 16, 5, 5 layer 2
            nn.Conv2d(16, 8, 3, stride=2, padding=1),  # b, 8, 3, 3 layer 3
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=1)  # b, 8, 2, 2
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 16, 3, stride=2),  # b, 16, 5, 5 layer 1
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 8, 5, stride=3, padding=1),  # b, 8, 15, 15 layer 2
            nn.ReLU(True),
            nn.ConvTranspose2d(8, 1, 2, stride=2, padding=1),  # b, 1, 28, 28 layer 3
            nn.Tanh()
        )

    # forward pass
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


Model Training

In [37]:
model = autoencoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate,
                             weight_decay=1e-5)

Gaussian noise

In [38]:
def add_noise(img, sigma):
    noise = torch.randn_like(img) * sigma
    noisy = img + noise
    return torch.clamp(noisy, -1., 1.)

Random noise level

In [39]:
sigma = random.choice([0.1, 0.2, 0.3])

Training loop

In [40]:
for epoch in range(num_epochs):
    total_loss = 0

    for data in dataloader:
        img, _ = data
        img = Variable(img)

        # Random noise each batch
        sigma = random.choice([0.1, 0.2, 0.3])
        noisy_img = add_noise(img, sigma)

        # Forward
        output = model(noisy_img)
        loss = criterion(output, img)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.data

    print('epoch [{}/{}], loss:{:.4f}'.format(epoch+1, num_epochs, total_loss))

    # =========================
    # Visualization every 10 epochs
    # =========================
    if epoch % 10 == 0:

        # -------- GRID (multiple images) --------
        batch_size_current = img.size(0)
        comparison = torch.cat([
            noisy_img,
            output,
            img
        ], dim=0)

        save_image(
            to_img(comparison.cpu().data),
            f'./results/grid_epoch_{epoch}.png',
            nrow=batch_size_current
        )

        # -------- SINGLE IMAGE --------
        sample = img[0].unsqueeze(0)
        noisy_sample = add_noise(sample, sigma)
        output_sample = model(noisy_sample)

        single = torch.cat([
            noisy_sample,
            output_sample,
            sample
        ])

        save_image(
            to_img(single.cpu().data),
            './results/single_epoch_{}.png'.format(epoch),
            nrow=3
        )


epoch [1/100], loss:101.3338
epoch [2/100], loss:54.3964
epoch [3/100], loss:47.3239
epoch [4/100], loss:43.7627
epoch [5/100], loss:41.7118
epoch [6/100], loss:40.3402
epoch [7/100], loss:39.3202
epoch [8/100], loss:38.5796
epoch [9/100], loss:38.0131
epoch [10/100], loss:37.4965
epoch [11/100], loss:37.0658
epoch [12/100], loss:36.7654
epoch [13/100], loss:36.3748
epoch [14/100], loss:36.1403
epoch [15/100], loss:35.8589
epoch [16/100], loss:35.6773
epoch [17/100], loss:35.4746
epoch [18/100], loss:35.2893
epoch [19/100], loss:35.1617
epoch [20/100], loss:35.0271
epoch [21/100], loss:34.9592
epoch [22/100], loss:34.7924
epoch [23/100], loss:34.7196
epoch [24/100], loss:34.5947
epoch [25/100], loss:34.4967
epoch [26/100], loss:34.5086
epoch [27/100], loss:34.3481
epoch [28/100], loss:34.4322
epoch [29/100], loss:34.2147
epoch [30/100], loss:34.1848
epoch [31/100], loss:34.2049
epoch [32/100], loss:34.0921
epoch [33/100], loss:33.9718
epoch [34/100], loss:33.9447
epoch [35/100], loss:3

Model saving 

Restarts

In [41]:
torch.save(model.state_dict(), './conv_autoencoder_week1.pth')

Incremental saving

In [42]:

# if os.path.exists('./autoencoder_week1.pth'):
#     model.load_state_dict(torch.load('./autoencoder_week1.pth'))
#     print("Model loaded, continuing training...")